# Interactive Decision Tree - Oracle ve Sample Data Demo

Bu notebook iki akisi gosterir:

1. Notebook icinde uretilen sample pandas `DataFrame` ile UI'i acmak.
2. Oracle'a farkli bir demo dataset yazmak, Oracle'dan geri okumak ve okunan datayi UI'da kullanmak.

Credential bilgileri `oracle_config/ora_config.ini` icinden okunur; kullanici/sifre ekrana yazdirilmaz ve `oracle_config/` git'e dahil edilmez.


## 0. Kurulum notu

Bu notebook'u repo kokunden calistiriyorsan once bir kez sunu calistir:

```powershell
.\.venv\Scripts\python.exe -m pip install -e ".[notebook,oracle]"
```

Notebook kernel'in bu `.venv` degilse, asagidaki `%pip install -e ".[notebook,oracle]"` satirini acip bir kez calistirabilirsin.


In [ ]:
# Import hatasi alirsan once kernel'in `interactive_decision_tree_env (.venv)` oldugunu kontrol et.
# Gerekirse bu satiri acip bir kez calistir:
# %pip install -e ".[notebook,oracle]"

import sys
from configparser import ConfigParser
from pathlib import Path
from urllib.parse import quote_plus

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("python_executable:", sys.executable)
print("project_root:", PROJECT_ROOT)

try:
    import numpy as np
    import pandas as pd
    from sqlalchemy import create_engine, text
    from interactive_decision_tree import launch_tree, launch_tree_sql
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "Notebook kernel'inde eksik paket var veya yanlis kernel secili. "
        "VS Code/Jupyter kernel olarak `interactive_decision_tree_env (.venv)` sec. "
        f"Aktif Python: {sys.executable}. Eksik modul: {exc.name}. "
        'Gerekirse bu hucreden once `%pip install -e \".[notebook,oracle]\"` calistir.'
    ) from exc


## 1. Notebook icinde sample DataFrame uretme

Bu bolum dis kaynaga baglanmadan RAM'de sample data uretir ve UI'a aktarir.


In [ ]:
rng = np.random.default_rng(7)
n = 500

sample_df = pd.DataFrame(
    {
        "age": rng.integers(21, 72, size=n),
        "income": rng.normal(52_000, 18_000, size=n).clip(12_000, 140_000).round(2),
        "tenure_months": rng.integers(0, 120, size=n),
        "segment": rng.choice(["A", "B", "C", "D", "E"], size=n),
        "channel": rng.choice(["branch", "mobile", "web", "call_center"], size=n),
        "region": rng.choice(["marmara", "ege", "akdeniz", "ic_anadolu"], size=n),
    }
)

sample_score = (
    (sample_df["income"] < 42_000).astype(int)
    + (sample_df["tenure_months"] < 18).astype(int)
    + sample_df["segment"].isin(["C", "D"]).astype(int)
    + (sample_df["channel"] == "mobile").astype(int)
    - (sample_df["age"] > 55).astype(int)
)
sample_df["risk_flag"] = np.where(sample_score >= 2, "high_risk", "low_risk")

sample_df.head()


In [ ]:
sample_url = launch_tree(
    sample_df,
    target="risk_flag",
    features=["age", "income", "tenure_months", "segment", "channel", "region"],
    session_name="Notebook sample data",
    port=8501,
    start_server=True,
    open_browser=True,
)
sample_url


## 2. Oracle baglantisini acma

Bu hucre `oracle_config/ora_config.ini` veya `ora_config/ora_config.ini` dosyasini okur, SQLAlchemy Oracle URL'ini bellekte olusturur ve `select 1 from dual` ile baglantiyi test eder.


In [ ]:
oracle_config_path = Path("oracle_config/ora_config.ini")
if not oracle_config_path.exists():
    oracle_config_path = Path("ora_config/ora_config.ini")
if not oracle_config_path.exists():
    raise FileNotFoundError("oracle_config/ora_config.ini bulunamadi.")

oracle_section = "ORA_PRD_ZTUSER"
parser = ConfigParser()
parser.read(oracle_config_path, encoding="utf-8")
if oracle_section not in parser:
    raise KeyError(f"INI icinde bolum bulunamadi: {oracle_section}")

cfg = parser[oracle_section]
oracle_url = (
    "oracle+oracledb://"
    f"{quote_plus(cfg['user'])}:{quote_plus(cfg['password'])}"
    f"@{cfg['host']}:{cfg.get('port', '1521')}/?service_name={quote_plus(cfg['service_name'])}"
)

engine = create_engine(oracle_url)
try:
    with engine.connect() as conn:
        ok_value = conn.execute(text("select 1 as ok from dual")).scalar_one()
finally:
    engine.dispose()

print({"section": oracle_section, "connection": "ok", "select_1": int(ok_value)})


## 3. Oracle icin farkli demo data uretme

Bu dataset lokal sample datadan farklidir. Sonraki hucrede Oracle'a tablo olarak yazilir.


In [ ]:
oracle_rng = np.random.default_rng(42)
oracle_n = 320

oracle_demo_df = pd.DataFrame(
    {
        "CUSTOMER_ID": np.arange(1, oracle_n + 1),
        "AGE": oracle_rng.integers(22, 76, size=oracle_n),
        "INCOME": oracle_rng.normal(64_000, 22_000, size=oracle_n).clip(15_000, 180_000).round(2),
        "TENURE_MONTHS": oracle_rng.integers(0, 144, size=oracle_n),
        "SEGMENT": oracle_rng.choice(["SME", "MASS", "AFFLUENT", "YOUNG"], size=oracle_n),
        "CHANNEL": oracle_rng.choice(["BRANCH", "MOBILE", "WEB", "CALL_CENTER"], size=oracle_n),
        "REGION": oracle_rng.choice(["MARMARA", "EGE", "AKDENIZ", "IC_ANADOLU", "KARADENIZ"], size=oracle_n),
        "UTILIZATION": oracle_rng.beta(2.2, 4.5, size=oracle_n).round(4),
    }
)

oracle_score = (
    (oracle_demo_df["INCOME"] < 48_000).astype(int)
    + (oracle_demo_df["TENURE_MONTHS"] < 24).astype(int)
    + oracle_demo_df["SEGMENT"].isin(["SME", "YOUNG"]).astype(int)
    + (oracle_demo_df["CHANNEL"] == "MOBILE").astype(int)
    + (oracle_demo_df["UTILIZATION"] > 0.52).astype(int)
    - (oracle_demo_df["AGE"] > 60).astype(int)
)
oracle_demo_df["RISK_FLAG"] = np.where(oracle_score >= 3, "high_risk", "low_risk")

oracle_demo_df.head()


## 4. Oracle'a yazma

Bu hucre mevcut kullanicinin schema'sinda `IDT_DEMO_TREE_DATA` tablosunu olusturur veya replace eder. Calistirmadan once bu tablo adinin senin ortaminda uygun oldugunu kontrol et.


In [ ]:
oracle_table_name = "IDT_DEMO_TREE_DATA"

engine = create_engine(oracle_url)
try:
    oracle_demo_df.to_sql(
        oracle_table_name,
        con=engine,
        if_exists="replace",
        index=False,
        chunksize=1000,
    )
    with engine.connect() as conn:
        row_count = conn.execute(text(f"select count(*) from {oracle_table_name}")).scalar_one()
finally:
    engine.dispose()

print({"table": oracle_table_name, "rows_written": int(row_count)})


## 5. Oracle'dan okuyup UI'da kullanma

Bu hucre datayi Oracle tablosundan geri cagirir ve `launch_tree_sql(...)` ile UI'a aktarir. UI tarafinda data snapshot olarak `.tree_sessions/` altina kaydedilir; sayfa yenilenince Oracle sorgusu tekrar calismaz.


In [ ]:
oracle_read_query = f"""
select
    CUSTOMER_ID,
    AGE,
    INCOME,
    TENURE_MONTHS,
    SEGMENT,
    CHANNEL,
    REGION,
    UTILIZATION,
    RISK_FLAG
from {oracle_table_name}
"""

oracle_ui_url = launch_tree_sql(
    oracle_url,
    query=oracle_read_query,
    target="RISK_FLAG",
    full_table=True,
    session_name="Oracle roundtrip demo data",
    port=8501,
    start_server=True,
    open_browser=True,
)
oracle_ui_url


## 6. Final agaci notebook'a geri yukleme

UI'da agaci finalize ettikten sonra en alttaki `Tree export` bolumunden `Download runnable tree pickle` ile dosyayi indir. Sonra ayni veya baska bir notebook'ta asagidaki gibi acabilirsin.

Not: Pickle dosyalarini sadece kendi urettigin guvenilir dosyalardan yukle.


In [ ]:
from interactive_decision_tree import load_tree_pickle, load_tree_json, score_tree_payload

# UI export pickle dosyasini Downloads klasorune indirdiysen:
tree_pickle_path = Path.home() / "Downloads" / "interactive_entropy_tree_runnable.pkl"
tree_payload = load_tree_pickle(tree_pickle_path)

print("loaded_tree_path:", tree_pickle_path)
print("loaded_tree_modified:", pd.Timestamp.fromtimestamp(tree_pickle_path.stat().st_mtime))
print("loaded_tree_nodes:", tree_payload.get("node_count"))

# JSON indirdiysen:
# tree_payload = load_tree_json(Path.home() / "Downloads" / "interactive_entropy_tree_runnable.json")

# tree_payload.keys()
# tree_payload["tree"]
# tree_payload["metrics"]


## 7. Tek musteri icin skorlama

Bu hucrede musteri degiskenlerini notebook icinde yaratip yukledigimiz gercek pickle/JSON payload ile skorlariz. `tree_payload` bir onceki hucrede UI exportundan yuklenmis olmalidir.

`prediction` global bir threshold'a gore degil, musteri hangi leaf'e dustuyse o leaf'in cogunluk sinifina gore gelir. Binary target icin `positive_class_probability` leaf icindeki positive class oranidir; `prediction_probability` tahmin edilen sinifin leaf icindeki oranidir.


In [ ]:
if "tree_payload" not in globals():
    raise RuntimeError("Once UI export pickle/JSON dosyasini tree_payload degiskenine yukle.")

payload_features = {str(feature) for feature in tree_payload.get("features", [])}

if {"AGE", "INCOME", "TENURE_MONTHS", "SEGMENT", "CHANNEL", "REGION", "UTILIZATION"}.issubset(payload_features):
    one_customer = {
        "CUSTOMER_ID": 999001,
        "AGE": 34,
        "INCOME": 35_000,
        "TENURE_MONTHS": 12,
        "SEGMENT": "SME",
        "CHANNEL": "MOBILE",
        "REGION": "MARMARA",
        "UTILIZATION": 0.62,
    }
else:
    one_customer = {
        "age": 34,
        "income": 35_000,
        "tenure_months": 12,
        "segment": "C",
        "channel": "mobile",
        "region": "marmara",
    }

score_result = score_tree_payload(tree_payload, one_customer)

print("prediction:", score_result["prediction"])
print("prediction_proba:", score_result["prediction_probability"])
print("positive_class:", score_result["positive_class"])
print("positive_class_proba:", score_result["positive_class_probability"])
print("leaf_node_id:", score_result["leaf_node_id"])
print("leaf_path:", score_result["leaf_path"])
if score_result.get("exported_leaf_path") != score_result.get("leaf_path"):
    print("exported_leaf_path:", score_result.get("exported_leaf_path"))
display(pd.DataFrame([score_result["class_probabilities"]]))
pd.DataFrame(score_result["trace"])
